# 9장: 어근추출(Lemmatization)을 통한 Ridge, LightGBM, XGBoost 모델링 (수정판)

이 노트북은 기존 09번 노트북에서 발생했던 세 가지 문제를 수정한 버전입니다.

**수정 사항**
1. `min_df=0.05`(비율) → `min_df=5`(정수, 최소 등장 횟수)로 수정 — 기존 설정은 148만 건의 5%(약 7만 4천 건) 이상 등장하는 단어만 남겨 어휘가 사실상 텅 비었습니다(실제 실행 결과 TF-IDF 컬럼이 35개뿐이었습니다).
2. 정형 피처를 `X_full`(995개 컬럼) 대신 `X_final`(297개, 유의변수만 추린 버전)로 교체 — dense 배열 변환 시 메모리 요구량을 큰 폭으로 줄입니다.
3. `X_full`에서는 텍스트(`text`) 컬럼만 추출해서 쓰고 나머지는 즉시 메모리에서 해제(`del` + `gc.collect()`)합니다.
4. LightGBM·XGBoost 학습 시 `.toarray()`로 dense 변환하지 않고 **sparse 행렬을 그대로** 넘깁니다 (두 라이브러리 모두 sparse 입력을 직접 지원합니다).
5. 코드에 원래 있던 문법 오류(닫히지 않은 괄호)를 수정했습니다.
6. **Optuna 하이퍼파라미터 튜닝은 Ridge에서만 진행**합니다. LightGBM·XGBoost는 대용량 sparse 데이터(5만+ 컬럼)에서 trial마다 학습 시간이 오래 걸리므로, 안정적인 성능을 내는 고정 파라미터로 바로 학습합니다. 학습 시간이 크게 단축됩니다.

**실행 전 확인해주세요**
- `../data/processed/X_full.pkl`, `X_final.pkl`, `y_log.pkl` 경로가 실제 파일 위치와 일치하는지 확인해주세요.
- 메모리 여유가 크지 않다면, 1단계(어근추출+TF-IDF 저장)까지 실행한 뒤 **커널을 재시작**하고 2단계부터 이어서 실행하는 것을 권장합니다(재시작 후 이어가는 방법은 노트북 맨 아래 별도 섹션에 정리했습니다).

## 1단계: 라이브러리 임포트

In [4]:
import gc
import numpy as np
import pandas as pd
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
import re
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
import lightgbm as lgb
import xgboost as xgb
import optuna
import joblib
from scipy.sparse import hstack, csr_matrix, save_npz, load_npz
import warnings
warnings.filterwarnings('ignore')

print("모든 라이브러리 임포트 완료!")

모든 라이브러리 임포트 완료!


c:\Users\mega\anaconda3\envs\ml-dev\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2단계: NLTK 데이터 다운로드

In [ ]:
print("NLTK 데이터 다운로드 중...")
nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
print("NLTK 데이터 다운로드 완료")

## 3단계: 텍스트 데이터 로드 (X_full에서 text 컬럼만 추출)

`X_full` 전체를 계속 들고 있지 않고, 필요한 `text` 컬럼만 뽑아낸 뒤 즉시 메모리에서 해제합니다.

In [ ]:
print("데이터 로드 중...")
X_full = pd.read_pickle("../data/processed/X_full.pkl")
print(f"X_full shape: {X_full.shape}")

# text 컬럼만 별도로 보관하고 X_full은 즉시 해제
text_series = X_full['text'].copy()
del X_full
gc.collect()

print(f"텍스트 데이터 개수: {len(text_series)}")
print(f"\n텍스트 샘플: {text_series.iloc[0][:100]}...")

## 4단계: 어근추출(Lemmatization) 적용

In [ ]:
lemmatizer = WordNetLemmatizer()

def lemmatize_text(text):
    """텍스트에 표제어 추출 적용"""
    if not isinstance(text, str):
        return ""

    tokens = word_tokenize(text.lower())

    lemmatized_tokens = []
    for token in tokens:
        # 숫자만 있거나 2글자 미만 제외
        if token.isdigit() or len(token) < 2:
            continue
        # 알파벳이 포함된 유의미한 단어만 표제어 추출
        if re.search('[a-zA-Z]', token):
            lemmatized_tokens.append(lemmatizer.lemmatize(token))

    return " ".join(lemmatized_tokens)

print("함수 정의 완료")

In [ ]:
print("텍스트 어근추출 적용 중... (약 10분 소요)")
tqdm.pandas()
text_lemmatized = text_series.progress_apply(lemmatize_text)

print("어근추출 완료")
print(f"\n원본 샘플: {text_series.iloc[0][:80]}...")
print(f"어근추출 샘플: {text_lemmatized.iloc[0][:80]}...")

# 원본 텍스트는 더 이상 필요 없으므로 해제
del text_series
gc.collect()

## 5단계: TF-IDF 벡터화 (수정: min_df를 정수로)

기존 코드는 `min_df=0.05`(비율)로 설정되어 있어 148만 건 기준 약 7만 4천 건 이상 등장하는 단어만 남기는 셈이었고, 그 결과 어휘가 35개까지 줄어들었습니다. `min_df=5`(정수, 즉 "최소 5개 문서에 등장")로 수정합니다.

In [ ]:
print("TF-IDF 벡터화 중...")
tfidf_vectorizer = TfidfVectorizer(
    max_features=50000,
    dtype=np.float32,
    stop_words='english',
    ngram_range=(1, 2),
    min_df=5,       # 수정: 0.05(비율) -> 5(정수, 최소 등장 문서 수)
    max_df=0.9,      # 수정: 0.85 -> 0.9 (지나치게 흔한 단어만 제외)
)

tfidf_lemmatized = tfidf_vectorizer.fit_transform(text_lemmatized)
print(f"TF-IDF 행렬 shape: {tfidf_lemmatized.shape}")

# 어근추출 결과와 벡터라이저는 더 이상 필요 없으므로 해제
del text_lemmatized
gc.collect()

## 6단계: TF-IDF 결과 저장 (중간 저장 지점)

여기까지 실행한 뒤, 메모리가 부족하다면 **커널을 재시작**하고 노트북 하단의 "커널 재시작 후 이어서 실행" 섹션부터 진행하세요. 저장해둔 파일만 불러오면 이어갈 수 있습니다.

In [ ]:
save_npz("../data/processed/tfidf_lemmatized_fixed.npz", tfidf_lemmatized)
print("TF-IDF 결과 저장 완료: ../data/processed/tfidf_lemmatized_fixed.npz")

## 7단계: 정형 피처 결합 (수정: X_full 대신 X_final 사용)

`X_full`(995개 컬럼)을 dense로 변환하면서 메모리 부족(11GB 할당 실패)이 발생했던 부분입니다. 이미 유의미한 변수만 선별해둔 `X_final`(297개 컬럼)을 사용하면 필요한 메모리가 3분의 1 이하로 줄어듭니다.

In [ ]:
print("정형 피처 로드 중...")
X_final = pd.read_pickle("../data/processed/X_final.pkl")
print(f"X_final shape: {X_final.shape}")

# 혹시 모를 타겟 누수 컬럼(price, log_price 등) 안전 점검
leak_cols = [col for col in X_final.columns if 'price' in col.lower()]
if leak_cols:
    print(f"타겟 누수 의심 컬럼 제거: {leak_cols}")
    X_final = X_final.drop(columns=leak_cols)

# float32로 변환 후 sparse화 (float64 대비 메모리 절반)
X_final_sparse = csr_matrix(X_final.values.astype(np.float32))
print(f"X_final_sparse shape: {X_final_sparse.shape}")

del X_final
gc.collect()

In [ ]:
print("정형 피처와 텍스트 피처 결합 중...")
tfidf_lemmatized = load_npz("../data/processed/tfidf_lemmatized_fixed.npz")

X_combined = hstack([X_final_sparse, tfidf_lemmatized]).tocsr()
print(f"결합된 특성 행렬 shape: {X_combined.shape}")

del X_final_sparse, tfidf_lemmatized
gc.collect()

## 8단계: 학습/테스트 분할 및 평가 함수 정의

In [ ]:
y_log = pd.read_pickle("../data/processed/y_log.pkl")

print("훈련/테스트 데이터 분할 중...")
X_train, X_test, y_train_log, y_test_log = train_test_split(
    X_combined, y_log, test_size=0.2, random_state=42
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

del X_combined
gc.collect()

In [ ]:
def rmsle(y_true, y_pred):
    """RMSLE 계산"""
    y_pred = np.clip(y_pred, 0, None)
    return np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(y_true))**2))

y_test_orig = np.expm1(y_test_log.values)
print("RMSLE 함수 정의 완료")

## 9단계: Ridge 회귀 - Optuna 하이퍼파라미터 튜닝

In [ ]:
def ridge_objective(trial):
    alpha = trial.suggest_float("alpha", 1e-3, 100.0, log=True)
    solver = trial.suggest_categorical("solver", ["auto", "sag", "sparse_cg"])

    model = Ridge(alpha=alpha, solver=solver, random_state=42, max_iter=10000)
    model.fit(X_train, y_train_log)

    pred_log = model.predict(X_test)
    pred = np.clip(np.expm1(pred_log), 0, None)

    return rmsle(y_test_orig, pred)

print("Ridge 목적 함수 정의 완료")

In [ ]:
print("=" * 80)
print("Ridge 회귀 - Optuna 하이퍼파라미터 튜닝 시작 (20 trials)")
print("=" * 80)

ridge_study = optuna.create_study(direction="minimize")
ridge_study.optimize(ridge_objective, n_trials=20, show_progress_bar=True)

print(f"\nRidge 최적 RMSLE: {ridge_study.best_value:.6f}")
print(f"Ridge 최적 파라미터: {ridge_study.best_params}")

In [ ]:
print("최종 Ridge 모델 훈련 중...")
best_ridge_params = ridge_study.best_params
ridge_model = Ridge(
    alpha=best_ridge_params["alpha"],
    solver=best_ridge_params["solver"],
    random_state=42,
    max_iter=10000
)
ridge_model.fit(X_train, y_train_log)

ridge_pred_log = ridge_model.predict(X_test)
ridge_pred = np.clip(np.expm1(ridge_pred_log), 0, None)
ridge_rmsle = rmsle(y_test_orig, ridge_pred)

# 과적합 여부 확인을 위한 train RMSLE
ridge_train_pred_log = ridge_model.predict(X_train)
ridge_train_pred = np.clip(np.expm1(ridge_train_pred_log), 0, None)
y_train_orig = np.expm1(y_train_log.values)
ridge_train_rmsle = rmsle(y_train_orig, ridge_train_pred)

print(f"Ridge Train RMSLE: {ridge_train_rmsle:.6f}")
print(f"Ridge Test  RMSLE: {ridge_rmsle:.6f}")

joblib.dump(ridge_model, "../data/processed/ridge_lemmatized_model_fixed.pkl")
print("Ridge 모델 저장 완료")

## 10단계: LightGBM - 고정 하이퍼파라미터로 학습 (Optuna 미사용)

**수정**: Optuna 튜닝은 Ridge에서만 진행하고, LightGBM은 일반적으로 안정적인 성능을 내는 고정 파라미터로 바로 학습합니다. 튜닝 단계가 빠지는 대신 학습이 훨씬 빨라집니다. `.toarray()`로 dense 변환하지 않고 `scipy.sparse` 행렬을 `lgb.Dataset`에 그대로 넘깁니다.

In [2]:
import multiprocessing
N_THREADS = multiprocessing.cpu_count()

train_data = lgb.Dataset(X_train, label=y_train_log.values)
valid_data = lgb.Dataset(X_test, label=y_test_log.values, reference=train_data)

print(f"LightGBM Dataset 준비 완료 (sparse 그대로 사용, 코어 {N_THREADS}개 사용)")

LightGBM Dataset 준비 완료 (sparse 그대로 사용, 코어 12개 사용)


In [3]:
lgb_params = {
    "objective": "regression",
    "metric": "rmse",
    "verbosity": -1,
    "boosting_type": "gbdt",
    "num_threads": N_THREADS,
    "force_col_wise": True,     # sparse 대용량에서 워밍업 스킵, 속도 향상
    "num_leaves": 63,
    "max_depth": 7,
    "learning_rate": 0.08,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 3,
    "min_child_samples": 20,
    "lambda_l1": 0.1,
    "lambda_l2": 0.1,
}

print("LightGBM 학습 중 (고정 파라미터)...")
lgb_model = lgb.train(
    lgb_params,
    train_data,
    valid_sets=[valid_data],
    num_boost_round=1000,
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)],
)

lgb_pred_log = lgb_model.predict(X_test, num_iteration=lgb_model.best_iteration)
lgb_pred = np.expm1(lgb_pred_log)
lgb_rmsle = rmsle(y_test_orig, lgb_pred)

print(f"LightGBM 최종 RMSLE: {lgb_rmsle:.6f}")

joblib.dump(lgb_model, "../data/processed/lightgbm_lemmatized_model_fixed.pkl")
print("LightGBM 모델 저장 완료")

LightGBM 학습 중 (고정 파라미터)...
LightGBM 최종 RMSLE: 0.503646
LightGBM 모델 저장 완료


## 11단계: XGBoost - 고정 하이퍼파라미터로 학습 (Optuna 미사용)

**수정**: XGBoost도 Optuna 튜닝 없이 고정 파라미터로 바로 학습합니다. sparse 행렬을 `xgb.DMatrix`에 직접 넘길 수 있어 `.toarray()`가 필요 없습니다.

In [3]:
dtrain = xgb.DMatrix(X_train, label=y_train_log.values)
dtest = xgb.DMatrix(X_test, label=y_test_log.values)

print("XGBoost DMatrix 준비 완료 (sparse 그대로 사용)")

XGBoost DMatrix 준비 완료 (sparse 그대로 사용)


In [4]:
xgb_params = {
    "objective": "reg:squarederror",
    "eval_metric": "rmse",
    "max_depth": 7,
    "learning_rate": 0.08,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "colsample_bylevel": 0.8,
    "min_child_weight": 5,
    "gamma": 0.1,
    "lambda": 1.0,
    "alpha": 0.1,
    "tree_method": "hist",
    "nthread": N_THREADS,
}

print("XGBoost 학습 중 (고정 파라미터)...")
xgb_model = xgb.train(
    xgb_params,
    dtrain,
    num_boost_round=1000,
    evals=[(dtest, "test")],
    early_stopping_rounds=50,
    verbose_eval=False
)

xgb_pred_log = xgb_model.predict(dtest)
xgb_pred = np.expm1(xgb_pred_log)
xgb_rmsle = rmsle(y_test_orig, xgb_pred)

print(f"XGBoost 최종 RMSLE: {xgb_rmsle:.6f}")

xgb_model.save_model("../models/xgboost_lemmatized_model_fixed.pkl")
print("XGBoost 모델 저장 완료")

XGBoost 학습 중 (고정 파라미터)...
XGBoost 최종 RMSLE: 0.503814
XGBoost 모델 저장 완료


In [5]:
if 'y_test_orig' not in globals():
    raise RuntimeError(
        "X_test / y_test_orig가 없습니다. 먼저 부록의 '커널 재시작 후 이어서 실행' 셀을 실행한 뒤 이 셀을 다시 실행하세요."
    )
 
if 'ridge_rmsle' not in globals():
    print("Ridge 결과가 메모리에 없어 저장된 모델을 불러옵니다...")
    ridge_model = joblib.load("../models/ridge_lemmatized_model_fixed.pkl")
    ridge_pred_log = ridge_model.predict(X_test)
    ridge_pred = np.clip(np.expm1(ridge_pred_log), 0, None)
    ridge_rmsle = rmsle(y_test_orig, ridge_pred)
    print(f"Ridge RMSLE (불러온 모델로 재계산): {ridge_rmsle:.6f}")
else:
    print(f"Ridge RMSLE (메모리에 있는 값 사용): {ridge_rmsle:.6f}")
 
if 'lgb_rmsle' not in globals():
    print("LightGBM 결과가 메모리에 없어 저장된 모델을 불러옵니다...")
    lgb_model = joblib.load("../models/lightgbm_lemmatized_model_fixed.pkl")
    lgb_pred_log = lgb_model.predict(X_test, num_iteration=lgb_model.best_iteration)
    lgb_pred = np.expm1(lgb_pred_log)
    lgb_rmsle = rmsle(y_test_orig, lgb_pred)
    print(f"LightGBM RMSLE (불러온 모델로 재계산): {lgb_rmsle:.6f}")
else:
    print(f"LightGBM RMSLE (메모리에 있는 값 사용): {lgb_rmsle:.6f}")
 
if 'xgb_rmsle' not in globals():
    print("XGBoost 결과가 메모리에 없어 저장된 모델을 불러옵니다...")
    xgb_model = xgb.Booster()
    xgb_model.load_model("../models/xgboost_lemmatized_model_fixed.pkl")
    dtest = xgb.DMatrix(X_test, label=y_test_log.values)
    xgb_pred_log = xgb_model.predict(dtest)
    xgb_pred = np.expm1(xgb_pred_log)
    xgb_rmsle = rmsle(y_test_orig, xgb_pred)
    print(f"XGBoost RMSLE (불러온 모델로 재계산): {xgb_rmsle:.6f}")
else:
    print(f"XGBoost RMSLE (메모리에 있는 값 사용): {xgb_rmsle:.6f}")
 
print("\n세 모델 결과 모두 준비 완료 — 12단계로 진행하세요.")
 

Ridge 결과가 메모리에 없어 저장된 모델을 불러옵니다...
Ridge RMSLE (불러온 모델로 재계산): 0.483642
LightGBM 결과가 메모리에 없어 저장된 모델을 불러옵니다...
LightGBM RMSLE (불러온 모델로 재계산): 0.503646
XGBoost RMSLE (메모리에 있는 값 사용): 0.503814

세 모델 결과 모두 준비 완료 — 12단계로 진행하세요.


## 12단계: 모델 성능 비교 및 결과 저장

In [6]:
print("=" * 80)
print("모델 성능 비교")
print("=" * 80)

results_df = pd.DataFrame({
    '모델': ['Ridge (Optuna 튜닝)', 'LightGBM (고정 파라미터)', 'XGBoost (고정 파라미터)'],
    'RMSLE': [ridge_rmsle, lgb_rmsle, xgb_rmsle]
})

print(results_df.to_string(index=False))

best_model_idx = results_df['RMSLE'].idxmin()
best_model_name = results_df.loc[best_model_idx, '모델']
best_rmsle = results_df.loc[best_model_idx, 'RMSLE']

print(f"\n최고 성능 모델: {best_model_name} (RMSLE: {best_rmsle:.6f})")

모델 성능 비교
                모델    RMSLE
 Ridge (Optuna 튜닝) 0.483642
LightGBM (고정 파라미터) 0.503646
 XGBoost (고정 파라미터) 0.503814

최고 성능 모델: Ridge (Optuna 튜닝) (RMSLE: 0.483642)


In [7]:
results_df.to_csv("../data/processed/lemmatization_results_fixed.csv", index=False)
print("결과 저장 완료: ../data/processed/lemmatization_results_fixed.csv")

print("\n" + "=" * 80)
print("9장(수정판) 모든 작업 완료!")
print("=" * 80)
print(f"\n저장된 모델:")
print(f"  1. Ridge: ridge_lemmatized_model_fixed.pkl")
print(f"  2. LightGBM: lightgbm_lemmatized_model_fixed.pkl")
print(f"  3. XGBoost: xgboost_lemmatized_model_fixed.pkl")
print(f"\n결과 요약:")
print(results_df.to_string(index=False))

결과 저장 완료: ../data/processed/lemmatization_results_fixed.csv

9장(수정판) 모든 작업 완료!

저장된 모델:
  1. Ridge: ridge_lemmatized_model_fixed.pkl
  2. LightGBM: lightgbm_lemmatized_model_fixed.pkl
  3. XGBoost: xgboost_lemmatized_model_fixed.pkl

결과 요약:
                모델    RMSLE
 Ridge (Optuna 튜닝) 0.483642
LightGBM (고정 파라미터) 0.503646
 XGBoost (고정 파라미터) 0.503814


---
## 부록: 커널 재시작 후 이어서 실행하기

6단계(TF-IDF 저장)까지 실행한 뒤 메모리 때문에 커널을 재시작했다면, 아래 셀부터 다시 실행하면 이어서 진행할 수 있습니다. 위쪽 1~6단계는 건너뛰어도 됩니다(단, 라이브러리 임포트 셀은 다시 실행해야 합니다).

In [1]:
# ===== 커널 재시작 후 여기부터 실행 =====
import gc
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
import lightgbm as lgb
import xgboost as xgb
import optuna
import joblib
from scipy.sparse import hstack, csr_matrix, load_npz
import warnings
warnings.filterwarnings('ignore')

# 7단계부터 이어서 실행
print("정형 피처 로드 중...")
X_final = pd.read_pickle("../data/processed/X_final.pkl")

leak_cols = [col for col in X_final.columns if 'price' in col.lower()]
if leak_cols:
    print(f"타겟 누수 의심 컬럼 제거: {leak_cols}")
    X_final = X_final.drop(columns=leak_cols)

X_final_sparse = csr_matrix(X_final.values.astype(np.float32))
del X_final
gc.collect()

print("텍스트 피처 로드 중...")
tfidf_lemmatized = load_npz("../data/processed/tfidf_lemmatized_fixed.npz")

X_combined = hstack([X_final_sparse, tfidf_lemmatized]).tocsr()
print(f"결합된 특성 행렬 shape: {X_combined.shape}")

del X_final_sparse, tfidf_lemmatized
gc.collect()

y_log = pd.read_pickle("../data/processed/y_log.pkl")

X_train, X_test, y_train_log, y_test_log = train_test_split(
    X_combined, y_log, test_size=0.2, random_state=42
)
del X_combined
gc.collect()

def rmsle(y_true, y_pred):
    y_pred = np.clip(y_pred, 0, None)
    return np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(y_true))**2))

y_test_orig = np.expm1(y_test_log.values)
print("이어서 실행할 준비 완료 - 이 노트북의 9단계(Ridge 튜닝)부터 순서대로 다시 실행하세요.")

c:\Users\mega\anaconda3\envs\ml-dev\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


정형 피처 로드 중...
텍스트 피처 로드 중...
결합된 특성 행렬 shape: (1481611, 50297)
이어서 실행할 준비 완료 - 이 노트북의 9단계(Ridge 튜닝)부터 순서대로 다시 실행하세요.
